# *Metodos de transformacion*

### preprocessing.KBinsDiscretizer
Convierte una variable numérica continua en bins (intervalos) discretos.
La idea principal de preprocessing.KBinsDiscretizer es tomar números continuos y convertirlos en grupos (intervalos).

Valores como 18, 22, 25, 35, 42 y 60 -->  Joven/Adulto/Mayor = 0/1/2

*Parametros clave*
- n_bins: cuántos intervalos crear
- encode: (onehot - En lugar de un número, crea varias columnas), (onehot-dense - Hace exactamente lo mismo que onehot, pero devuelve un arreglo 'matriz' normal en lugar de una matriz dispersa 'sparse') o (ordinal - Devuelve un numero.) --> (cómo se representa el resultado)
- strategy: ('quantile' - es importante es que cada grupo tenga aproximadamente la misma cantidad de datos.) ('kmeans' - grupos donde los valores sean parecidos entre sí.)

##### trabaja sobre los datos originales (edad, salario, área, población, etc.).

### Flujo 
Datos originales
        │
        ▼
Seleccionar columna (ej. population)
        │
        ▼
KBinsDiscretizer(n_bins, strategy)
        │
        ▼
Datos discretizados (bins ordinales/categoricos)
        │
        ▼
Mapear bins a etiquetas (opcional)
        │
        ▼
Variable categorica lista para EDA / modelos

In [6]:
# cargar csv y librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
from scipy import stats

dataset = '../data/countries.csv'
df = pd.read_csv(dataset, sep=";")

df[['name', 'population']].head(10)

,name,population
0,Andorra,84000
1,United Arab Emirates,4975593
2,Afghanistan,29121286
3,Antigua and Barbuda,86754
4,Anguilla,13254
5,Albania,2986952
6,Armenia,2968000
7,Angola,13068161
8,Antarctica,0
9,Argentina,41343201


In [8]:
# USANDO KBinsDiscretizer 
from sklearn.preprocessing import KBinsDiscretizer

kbd = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
df['population_category'] = kbd.fit_transform(df[['population']])

df[['name', 'population', 'population_category']].head(10)

/home/meme/Documentos/9-IDGS/ExtraccionDB/.venv/lib64/python3.14/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,name,population,population_category
0,Andorra,84000,0.0
1,United Arab Emirates,4975593,2.0
2,Afghanistan,29121286,3.0
3,Antigua and Barbuda,86754,0.0
4,Anguilla,13254,0.0
5,Albania,2986952,1.0
6,Armenia,2968000,1.0
7,Angola,13068161,2.0
8,Antarctica,0,0.0
9,Argentina,41343201,3.0


### KernelCenterer
Centra una matriz de similitud (kernel) para que su promedio sea 0, crea una matriz que indica qué tan parecidos son los datos entre sí.

*Parametros clave*
fit(): aprende como centrar la matriz.
transform(): aplica el centrado.
fit_transform(): aprende y centra la matriz en un solo paso.


##### trabaja sobre una matriz de similitud ya calculada, donde cada valor indica que tan parecidos son datos 

### Flujo 

Datos originales
        │
        ▼
Seleccionar columnas (ej. area, population)
        │
        ▼
StandardScaler
        │
        ▼
Datos estandarizados
        │
        ▼
rbf_kernel() - kernel gaussiano(que tan similares son)
        │
        ▼
K (Matriz de similitud)
        │
        ▼
KernelCenterer()
        │
        ▼
K centrada
        │
        ▼
Kernel PCA, SVM, etc.


In [ ]:
from sklearn.preprocessing import  KernelCenterer, StandardScaler
from sklearn.metrics.pairwise import rbf_kernel

X = df[['area', 'population']].dropna()
X_scaled = StandardScaler().fit_transform(X)
K = rbf_kernel(X_scaled, gamma=0.1)

# Valores de similitud entre 0 y 1, pero no centrados (el promedio de las similitudes no es cero, lo que puede sesgar algoritmos posteriores como Kernel PCA).
print(K[:3, :3])

[[1.         0.99963487 0.98232716]
 [0.99963487 1.         0.98694544]
 [0.98232716 0.98694544 1.        ]]


In [ ]:
# transformacion
kc = KernelCenterer()
K_centered = kc.fit_transform(K)

# Ahora los valores pueden ser positivos o negativos, y la matriz está centrada (la media de cada fila/columna en el espacio de características es cero) -- KernelPCA
print(K_centered[:3, :3])

[[ 0.0096283   0.00741385 -0.00679632]
 [ 0.00741385  0.00592966 -0.00402736]
 [-0.00679632 -0.00402736  0.01212473]]
